In [178]:
import pandas as pd
import requests

import re, json

rq = requests.Session()

In [179]:
url = lambda x: f'https://datasets-server.huggingface.co/rows?dataset=emozilla%2Fsat-reading&config=default&split=train&offset={100 * x}&length=100'

In [180]:
def get(url: str):
    pre = rq.get(url).text
    f = json.loads(pre)
    v = [f['rows'][i]['row'] for i in range(len(f['rows']))]
    return pd.DataFrame(v)

In [181]:
ans = pd.DataFrame()
now = 0

while True:
    tmp = get(url(now))
    now += 1
    if tmp.empty:
        break
    ans = pd.concat([ans, tmp])

ans = ans.reset_index()
ans

,index,text,answer,requires_line,id
0,0,SAT READING COMPREHENSION TEST\n\nThis passage...,A,False,sat-practice_7-question_3
1,1,SAT READING COMPREHENSION TEST\n\nThis passage...,D,True,sat-practice_7-question_10
2,2,SAT READING COMPREHENSION TEST\n\nThis passage...,C,True,sat-practice_7-question_7
3,3,SAT READING COMPREHENSION TEST\n\nThis passage...,D,False,sat-practice_7-question_1
4,4,SAT READING COMPREHENSION TEST\n\nThis passage...,B,True,sat-practice_7-question_4
...,...,...,...,...,...
293,93,SAT READING COMPREHENSION TEST\n\nThis passage...,C,False,sat-practice_5-question_42
294,94,SAT READING COMPREHENSION TEST\n\nThis passage...,D,True,sat-practice_5-question_44
295,95,SAT READING COMPREHENSION TEST\n\nThis passage...,B,True,sat-practice_5-question_52
296,96,SAT READING COMPREHENSION TEST\n\nThis passage...,C,True,sat-practice_5-question_48


In [182]:
ans.to_csv('ds.csv', index = False)

In [183]:
filter = re.compile(r'^SAT READING COMPREHENSION TEST\n*(.+)\n*Question (\d+):\n*(.+)\n*Answer:$', flags = re.MULTILINE | re.DOTALL)

In [184]:
filter.findall(ans.iloc[29].text)

[('Passage 1 is adapted from Alexis de Tocqueville, Democracy\nin America, Volume 2. Originally published in 1840. Passage 2\nis adapted from Harriet Taylor Mill, “Enfranchisement of\nWomen.” Originally published in 1851. As United States and\nEuropean societies grew increasingly democratic during the\nnineteenth century, debates arose about whether freedoms\nenjoyed by men should be extended to women as well.\n\nPassage 1\n    I have shown how democracy destroys or\nmodifies the different inequalities which originate in\nsociety; but is this all? or does it not ultimately affect\nthat great inequality of man and woman which has\nseemed, up to the present day, to be eternally based\nin human nature? I believe that the social changes\nwhich bring nearer to the same level the father and\nson, the master and servant, and superiors and\ninferiors generally speaking, will raise woman and\nmake her more and more the equal of man. But here,\nmore than ever, I feel the necessity of making myse

In [185]:
ans['re'] = ans.text.map(lambda x: filter.findall(x))

In [186]:
ans['paragraph'] = ans.re.map(lambda x: x[0][0])
ans['qid'] = ans.re.map(lambda x: x[0][1])
ans['question'] = ans.re.map(lambda x: x[0][2])

In [187]:
id_filter = re.compile(r'^sat-practice_(\d+)-question_(\d+)$')

In [188]:
ans['re'] = ans.id.map(lambda x: id_filter.findall(x))

ans['pid'] = ans.re.map(lambda x: x[0][0])
ans['tqid'] = ans.re.map(lambda x: x[0][1])

assert(all(ans.qid == ans.tqid))

In [189]:
ans['gid'] = pd.to_numeric(ans.pid)
ans['qid'] = pd.to_numeric(ans.qid)

In [190]:
ans = ans.sort_values(['gid', 'qid'])
ans = ans.drop(['re', 'tqid', 'index'], axis = 1)

In [191]:
ans = ans.reset_index(drop = True)

In [192]:
mapper = {i: j + 1 for j, i in enumerate(ans.paragraph.unique())}

In [193]:
ans['pid'] = ans.paragraph.map(lambda x: mapper[x])

In [194]:
assert len(ans.groupby('pid').paragraph.unique().map(lambda x: len(x)).unique()) == 1

In [195]:
ans

,text,answer,requires_line,id,paragraph,qid,question,pid,gid
0,SAT READING COMPREHENSION TEST\n\nThis passage...,B,False,sat-practice_1-question_1,"This passage is from Lydia Minatoya, The Stran...",1,Which choice best describes what happens in th...,1,1
1,SAT READING COMPREHENSION TEST\n\nThis passage...,B,False,sat-practice_1-question_2,"This passage is from Lydia Minatoya, The Stran...",2,Which choice best describes the developmental ...,1,1
2,SAT READING COMPREHENSION TEST\n\nThis passage...,C,False,sat-practice_1-question_3,"This passage is from Lydia Minatoya, The Stran...",3,"As used in line 1 and line 65, “directly” most...",1,1
3,SAT READING COMPREHENSION TEST\n\nThis passage...,A,False,sat-practice_1-question_4,"This passage is from Lydia Minatoya, The Stran...",4,Which reaction does Akira most fear from Chie?...,1,1
4,SAT READING COMPREHENSION TEST\n\nThis passage...,D,False,sat-practice_1-question_6,"This passage is from Lydia Minatoya, The Stran...",6,"In the passage, Akira addresses Chie with\nA) ...",1,1
...,...,...,...,...,...,...,...,...,...
293,SAT READING COMPREHENSION TEST\n\nThis passage...,D,False,sat-practice_8-question_47,"This passage is adapted from Daniel Chamovitz,...",47,"According to the passage, which statement best...",40,8
294,SAT READING COMPREHENSION TEST\n\nThis passage...,B,False,sat-practice_8-question_48,"This passage is adapted from Daniel Chamovitz,...",48,Which choice describes a scenario in which Hod...,40,8
295,SAT READING COMPREHENSION TEST\n\nThis passage...,B,True,sat-practice_8-question_49,"This passage is adapted from Daniel Chamovitz,...",49,"As used in line 67, “demonstrated” most nearly...",40,8
296,SAT READING COMPREHENSION TEST\n\nThis passage...,B,False,sat-practice_8-question_50,"This passage is adapted from Daniel Chamovitz,...",50,"Based on the passage, what potential criticism...",40,8


In [ ]:
ans.to_csv('ds.csv', index = False)